In [1]:
import csv
import pandas as pd
import numpy as np
from emmaemb.core import Emma

DATA_PATH = '/home/unix/vkrhk/EmmaEmb/data/finetuned-embeddings'
CSV_FILE = f'{DATA_PATH}/test.txt'
EMBEDDINGS_PATH = f'{DATA_PATH}/finetuned-650M-embeddings'

feature_data = []
embeddings = []

# Amino acid classification dictionary
amino_acid_classification = {
    'V': 'hydrophobic',
    'L': 'hydrophobic',
    'I': 'hydrophobic',
    'M': 'hydrophobic',
    'F': 'hydrophobic',
    'W': 'tryptophan',
    'P': 'proline',
    'S': 'polar uncharged',
    'T': 'polar uncharged',
    'C': 'Cytosine',
    'Y': 'Tyrosine',
    'N': 'polar uncharged',
    'Q': 'polar uncharged',
    'D': 'charged',
    'E': 'charged',
    'K': 'charged',
    'R': 'charged',
    'A': 'Alanine',
    'H': 'Histidine',
    'G': 'Glycine',
    'X': 'Unknown',
    'U': 'Selenocysteine'
}

embeddings_name = 'FINETUNED-650M-ESM2'
with open(CSV_FILE, 'r') as f:
    reader = csv.reader(f, delimiter=';')
    for ii, row in enumerate(reader):
        protein_id = row[0] + row[1]
        annotation = row[3].split(' ')
        annotation = [int(i[1:]) for i in annotation]
        sequence = row[4]

        embedding = np.load(f'{EMBEDDINGS_PATH}/{protein_id}.npy')        

        for i in range(len(sequence)):
            feature_data.append([f'{protein_id}_{i}', sequence[i], amino_acid_classification[sequence[i]], 'CRYPTIC-BINDING' if i in annotation else 'NON-BINDING'])

        embeddings.append(embedding)

concatenated_embeddings_path = f"{DATA_PATH}/{embeddings_name}_binding_site_embeddings.npy"
embeddings = np.concatenate(embeddings, axis=0)
np.save(concatenated_embeddings_path, embeddings)  

feature_data = pd.DataFrame.from_records(feature_data, columns=["ID", "amino acid", "classification", "binding_site"])

emma = Emma(feature_data=feature_data)
emma.add_emb_space(
    embeddings_source=concatenated_embeddings_path,
    emb_space_name=embeddings_name)


55572 samples loaded.
Categories in meta data: ['amino acid', 'classification', 'binding_site']
Numerical columns in meta data: []
Embedding space 'FINETUNED-650M-ESM2' added successfully.
Embeddings have 1280 features each.


In [8]:
from emmaemb.vizualisation import (
    plot_emb_space,
    plot_knn_alignment_scores_across_k_and_distance_metrics,
    plot_pairwise_distance_comparison
)
fig_1 = plot_emb_space(emma, emb_space=embeddings_name, color_by="binding_site", method="PCA")
fig_1.update_traces(marker={'size': 4}, opacity=.3)
fig_1.update_layout(legend=dict(itemsizing='constant'), legend_x=0, legend_y=1)
fig_1.update_layout(height=600, width=800)
fig_1.show()

In [5]:
from emmaemb.vizualisation import plot_knn_alignment_across_classes

emma.calculate_pairwise_distances(emb_space=embeddings_name, metric='euclidean')

fig_3 = plot_knn_alignment_across_classes(
    emma, k=3, metric='euclidean', feature='binding_site')
fig_3.update_layout(height=400, width=700)
fig_3.show()


Pairwise distances using euclidean already calculated.
